# Pawnee National Grassland Land Swap
- **Objective:** For this notebook, the boundary of the grassland is prepared for later use in downstream notebooks
- **Author:** Max Warnock
- **Code review and/or edits:** Kayleigh Ward
- **Date:** April 9, 2026
- **Last date of revision:** April 16, 2026

---
### 🛠️ Prerequisites & Setup
* **Libraries:** ex: `pandas`, `scikit-learn`, `matplotlib`
* **Data Sources:** ex: `/data/raw/dataset.csv`
* **Related Notebooks:** ex: [Link to previous step]

### 🏗️ Methodology
1.  **Data Loading:** ex: Read from SQL
2.  **Cleaning:** ex: Handle outliers
3.  **Visualization:** ex: Plot distributions

### ⚡ Troubleshooting/Notes
* Example Note: If data updates, run `01_refresh.ipynb` first.

---


### Import libraries

In [1]:
### file paths, OS operations, utilities
import os
import pathlib
import zipfile
import time
from glob import glob
from getpass import getpass

### data handling 
import pandas as pd
import geopandas as gpd

### web requests / data download
import requests

### geospatial visualization 
import holoviews as hv
import hvplot.pandas
import cartopy.crs as ccrs

### GBIF API access
import pygbif.occurrences as occ
import pygbif.species as species

### Set up file path structure

In [2]:
### set up root file path
# Walk up from the current directory to find the repo root (contains .git)
_cwd = pathlib.Path(os.getcwd()).resolve()
repo_root = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / '.git').exists()),
    _cwd
)
os.chdir(repo_root)

data_dir = os.path.join(repo_root, 'data')
os.makedirs(data_dir, exist_ok=True)

print(f'Repo root: {repo_root}')

Repo root: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project


### Inner directories for data processing

In [3]:
### set a directory for the National Grassland boundary data
boundary_dir = os.path.join(data_dir, 'boundaries')
os.makedirs(boundary_dir, exist_ok=True)


### set an inner directory for boundary data processing
bound_process_dir = os.path.join(boundary_dir, 'boundary-data-processing')
os.makedirs(bound_process_dir, exist_ok=True)


### set a directory for the master boundary data
master_boundary_dir = os.path.join(bound_process_dir, 'master_boundary')
os.makedirs(master_boundary_dir, exist_ok=True)


### set a directory for the federal boundary data
federal_boundary_dir = os.path.join(bound_process_dir, 'federal_boundary')
os.makedirs(federal_boundary_dir, exist_ok=True)


### set a directory for the state boundary data
state_boundary_dir = os.path.join(bound_process_dir, 'state_boundary')
os.makedirs(state_boundary_dir, exist_ok=True)

### Master boundary data download

In [4]:
### Google Drive url
master_boundary_url = f"https://drive.google.com/uc?export=download&id=1gsL1tzZu6ZH28b3PQm4VgZzlY2GrkO8p"

### zip path
download_path = os.path.join(master_boundary_dir, "pawnee_master_boundary.zip")

### create a session
session = requests.Session()

### request the file
response = session.get(master_boundary_url, stream=True)
response.raise_for_status()

### save the zip file
with open(download_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)

print(f"Downloaded to: {download_path}")

### unzip the file
with zipfile.ZipFile(download_path, "r") as zip_ref:
    zip_ref.extractall(master_boundary_dir)

print(f"Extracted to: {master_boundary_dir}")

Downloaded to: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project\data\boundaries\boundary-data-processing\master_boundary\pawnee_master_boundary.zip
Extracted to: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project\data\boundaries\boundary-data-processing\master_boundary


In [5]:
### path to extracted folder
master_boundary_dir = os.path.join(master_boundary_dir, "pawnee_master_boundary")

### build full path
pawnee_master_boundary_path = os.path.join(master_boundary_dir, "pawnee_master_boundary.shp")

### read the shapefile
pawnee_master_boundary_gdf = gpd.read_file(pawnee_master_boundary_path)

### check it
pawnee_master_boundary_gdf.head()

,Id,Location,geometry
0,0,1.0,"POLYGON ((-11593123.97 5012571.209, -11542474...."
1,0,NaN,"POLYGON ((-11614609.509 4955044.708, -11616750..."


In [6]:
### make sure its projected in EPSG 4326
pawnee_master_boundary_gdf = pawnee_master_boundary_gdf.to_crs(epsg=4326)

### plot with hvplot
pawnee_master_boundary_gdf.hvplot(
    geo=True,
    tiles="EsriImagery",
    title="Pawnee National Grassland Boundaries",
    line_color="white",
    line_width=2,
    fill_alpha=0,
    width=600,
    height=500
)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

### Download federal parcel data

In [7]:
### set the data download URL
fed_boundary_url = "https://drive.google.com/uc?export=download&id=1XucJkr6SKhwxGqMfoN8pBbFcNHlEk7-7"

### path where zip will be saved
fed_boundary_path = os.path.join(federal_boundary_dir, "fed_grassland.zip")

### download file
response = requests.get(fed_boundary_url, stream=True)
response.raise_for_status()

with open(fed_boundary_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

print(f"Downloaded to: {fed_boundary_path}")

Downloaded to: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project\data\boundaries\boundary-data-processing\federal_boundary\fed_grassland.zip


In [8]:
### unzip the file
with zipfile.ZipFile(fed_boundary_path, 'r') as zip_ref:
    zip_ref.extractall(federal_boundary_dir)

print(f"Extracted to: {federal_boundary_dir}")

Extracted to: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project\data\boundaries\boundary-data-processing\federal_boundary


In [9]:
### path to extracted folder
extrct_fed_boundary_dir = os.path.join(federal_boundary_dir, "fed_grassland")

### build full path
pawnee_fed_boundary_path = os.path.join(extrct_fed_boundary_dir, "fed_grasslands_reproject.shp")

### read the file
fed_boundary_gdf = gpd.read_file(pawnee_fed_boundary_path)

### check it
fed_boundary_gdf.head()

,NATIONALGR,GRASSLANDN,GIS_ACRES,SHAPE_AREA,SHAPE_LEN,geometry
0,295510010328,Cedar River National Grassland,6717.517,0.003157,0.999947,"MULTIPOLYGON (((-11334797.343 5773775.96, -113..."
1,295509010328,Sheyenne National Grassland,70428.175,0.033356,4.097398,"MULTIPOLYGON (((-10832591.139 5863197.121, -10..."
2,295520010328,Little Missouri National Grassland,1025335.132,0.492756,60.369307,"MULTIPOLYGON (((-11579107.712 6027028.778, -11..."
3,295517010328,Grand River National Grassland,154665.337,0.072336,9.789150,"MULTIPOLYGON (((-11363652.936 5730203.348, -11..."
4,295522010328,Comanche National Grassland,444413.905,0.183064,26.658022,"MULTIPOLYGON (((-11579747.667 4536041.62, -115..."


In [10]:
### select only Pawnee National Grassland
pawnee_fed = fed_boundary_gdf[fed_boundary_gdf["GRASSLANDN"] == "Pawnee National Grassland"].copy()

### check the result
pawnee_fed

,NATIONALGR,GRASSLANDN,GIS_ACRES,SHAPE_AREA,SHAPE_LEN,geometry
5,295523010328,Pawnee National Grassland,208424.885,0.089972,15.341594,"MULTIPOLYGON (((-11641911.317 4986805.57, -116..."


In [11]:
### make sure its in EPSG 4326
pawnee_fed = pawnee_fed.to_crs(epsg=4326)

### plot with hvplot
pawnee_fed.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    tiles="EsriImagery",
    title="Pawnee National Grassland Federal Boundaries",
    line_color="white",
    line_width=1,
    fill_alpha=0,
    width=600,
    height=500
)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

### Download state parcel data

In [12]:
### url
state_boundary_url = "https://services.arcgis.com/ewjSqmSyHJnkfBLL/arcgis/rest/services/Parcels_open_data/FeatureServer/replicafilescache/Parcels_open_data_542388106199933781.zip"

### local zip path
zip_path = os.path.join(state_boundary_dir, "state_parcels.zip")

### download
response = requests.get(state_boundary_url, stream=True)
response.raise_for_status()

with open(zip_path, "wb") as f:
    for chunk in response.iter_content(8192):
        if chunk:
            f.write(chunk)

print("Downloaded:", zip_path)

Downloaded: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project\data\boundaries\boundary-data-processing\state_boundary\state_parcels.zip


In [13]:
### unzip
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(state_boundary_dir)

print("Extracted to:", state_boundary_dir)

Extracted to: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project\data\boundaries\boundary-data-processing\state_boundary


In [14]:
### path to extracted file
state_boundary_path = os.path.join(state_boundary_dir, "Parcels_open_data.shp")

### read the file
state_boundary_gdf = gpd.read_file(state_boundary_path)

### check it
state_boundary_gdf.head()

,PARCEL,MHSPACE,ACCOUNTTYP,ACCOUNTNO,NAME,ADDRESS1,ADDRESS2,CITY,STATE,ZIPCODE,...,Shape_Leng,RECEPTION_,AddressPre,LGLANDASD,LGIMPASD,TOTALLGASD,SCLANDASD,SCIMPASD,TOTALSCASD,geometry
0,002919000001,NaN,R,R0000186,GREEN ELSIE KLINGINSMITH (1/3 INT),530 MCKINLEY ST,NaN,STERLING,CO,807512637,...,18720.439972,NaN,None,1940.0,0.0,1940.0,1940.0,0.0,1940.0,"POLYGON ((-11541970.912 5012592.769, -11540907..."
1,002919000002,NaN,R,R0000286,U S A,2850 YOUNGFIELD ST,NaN,LAKEWOOD,CO,802157210,...,1320.086368,NaN,None,10420.0,0.0,10420.0,10420.0,0.0,10420.0,"POLYGON ((-11542213.325 5011302.221, -11542213..."
2,002920000003,NaN,R,R0000386,GREEN ELSIE KLINGINSMITH (1/3 INT),530 MCKINLEY ST,NaN,STERLING,CO,807512637,...,17354.419416,NaN,None,1970.0,0.0,1970.0,1970.0,0.0,1970.0,"POLYGON ((-11538243.903 5012270.018, -11538249..."
3,002921000001,NaN,R,R0000586,GREEN ELSIE KLINGINSMITH (1/3 INT),530 MCKINLEY ST,NaN,STERLING,CO,807512637,...,17300.695352,NaN,None,2140.0,0.0,2140.0,2140.0,0.0,2140.0,"POLYGON ((-11536130.965 5012279.379, -11536129..."
4,002922000002,NaN,R,R0000686,SHEFFLER RANCH LLC,65295 COUNTY ROAD 135,NaN,NEW RAYMER,CO,807429645,...,17239.487033,NaN,None,1930.0,8430.0,10360.0,1930.0,9130.0,11060.0,"POLYGON ((-11534006.724 5012304.442, -11534006..."


In [15]:
### select only state owned parcels
pawnee_state = state_boundary_gdf[state_boundary_gdf["NAME"] == "COLORADO STATE OF"].copy()

### check the result
pawnee_state

,PARCEL,MHSPACE,ACCOUNTTYP,ACCOUNTNO,NAME,ADDRESS1,ADDRESS2,CITY,STATE,ZIPCODE,...,Shape_Leng,RECEPTION_,AddressPre,LGLANDASD,LGIMPASD,TOTALLGASD,SCLANDASD,SCIMPASD,TOTALSCASD,geometry
14,002928000006,NaN,R,R0002586,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,15825.837726,NaN,None,59520.0,0.0,59520.0,59520.0,0.0,59520.0,"POLYGON ((-11536129.85 5010677.712, -11536132...."
22,002934000004,NaN,R,R0004386,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,15821.058075,NaN,None,42900.0,0.0,42900.0,42900.0,0.0,42900.0,"POLYGON ((-11534015.184 5008563.732, -11534016..."
24,002936000004,NaN,R,R0004586,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,21096.936781,NaN,None,79580.0,0.0,79580.0,79580.0,0.0,79580.0,"POLYGON ((-11529751.924 5008576.317, -11529766..."
60,003136000004,NaN,R,R0009086,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,21197.217066,NaN,None,2860.0,0.0,2860.0,2860.0,0.0,2860.0,"POLYGON ((-11542463.578 5008481.775, -11542454..."
134,003536000005,NaN,R,R0018186,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,20812.895098,NaN,None,84780.0,0.0,84780.0,84780.0,0.0,84780.0,"POLYGON ((-11567892.461 5008243.702, -11567882..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162670,147536000016,NaN,R,R6507186,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,15792.583824,NaN,None,157740.0,0.0,157740.0,157740.0,0.0,157740.0,"POLYGON ((-11631752.413 4867090.685, -11632275..."
163042,147723000008,NaN,R,R6528086,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,7923.273242,NaN,None,10930.0,0.0,10930.0,10930.0,0.0,10930.0,"POLYGON ((-11622283.992 4870272.892, -11622809..."
163176,147736000006,NaN,R,R6534286,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,21125.739779,NaN,None,199430.0,0.0,199430.0,199430.0,0.0,199430.0,"POLYGON ((-11619149.069 4868145.656, -11619151..."
163225,147909000003,NaN,R,R6538086,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,21102.642804,NaN,None,85320.0,0.0,85320.0,85320.0,0.0,85320.0,"POLYGON ((-11612812.629 4876584.618, -11612814..."


In [16]:
### make sure its projected in EPSG 4326
pawnee_state = pawnee_state.to_crs(epsg=4326)

### clip to the master boundary
pawnee_state = gpd.clip(
    pawnee_state,
    pawnee_master_boundary_gdf
)

### plot with hvplot
pawnee_state.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Pawnee National Grassland State Boundaries',
    fill_color = None,
    line_color = "white",
    frame_width = 600
)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

In [17]:
### state plot
state_plot = pawnee_state.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    color="blue",
    fill_alpha=0.4,
    line_color="white",
    line_width=1,
    label="State"
)

### federal plot
fed_plot = pawnee_fed.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    color="green",
    fill_alpha=0.4,
    line_color="white",
    line_width=1,
    label="Federal"
)

### master boundary plot
master_plot = pawnee_master_boundary_gdf.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    fill_alpha=0,
    line_color="white",
    line_width=2,
    label="Master Boundary"
)

### create the plot
(hv.element.tiles.EsriImagery() * state_plot * fed_plot * master_plot
).opts(
    title="Pawnee State, Federal, and Master Boundaries",
    width=700,
    height=550,
    legend_position="top_left"
)

:Overlay
   .Tiles.I                  :Tiles   [x,y]
   .Polygons.State           :Polygons   [Longitude,Latitude]
   .Polygons.Federal         :Polygons   [Longitude,Latitude]
   .Polygons.Master_Boundary :Polygons   [Longitude,Latitude]

### Select only the Western Pawnee area

In [18]:
### select only western pawnee area
pawnee_west = pawnee_master_boundary_gdf[pawnee_master_boundary_gdf["Location"].isna()].copy()
pawnee_west

,Id,Location,geometry
1,0,NaN,"POLYGON ((-104.33581 40.6104, -104.35504 40.61..."


In [19]:
### clip state to the western pawnee
pawnee_state_west = gpd.clip(
    pawnee_state,
    pawnee_west
)

In [20]:
### clip fed to the western pawnee
pawnee_fed_west = gpd.clip(
    pawnee_fed,
    pawnee_west
)

In [21]:
### plot all three again for the western pawnee

### state plot
state_plot_west = pawnee_state_west.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    color="blue",
    fill_alpha=0.4,
    line_color="white",
    line_width=1,
    label="State"
)

### federal plot
fed_plot_west = pawnee_fed_west.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    color="green",
    fill_alpha=0.4,
    line_color="white",
    line_width=1,
    label="Federal"
)

### master boundary plot west
master_plot_west = pawnee_west.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    fill_alpha=0,
    line_color="white",
    line_width=2,
    label="Master Boundary"
)

### create the plot
(hv.element.tiles.EsriImagery() * state_plot_west * fed_plot_west * master_plot_west
).opts(
    title="West Pawnee State, Federal, and Master Boundaries",
    width=700,
    height=550,
    legend_position="top_left"
)

:Overlay
   .Tiles.I                  :Tiles   [x,y]
   .Polygons.State           :Polygons   [Longitude,Latitude]
   .Polygons.Federal         :Polygons   [Longitude,Latitude]
   .Polygons.Master_Boundary :Polygons   [Longitude,Latitude]

### Create file structure for Pawnee West processed data

In [22]:
### set a directory for final boundary data
bound_final_dir = os.path.join(boundary_dir, 'boundary-data-final-west')
os.makedirs(bound_final_dir, exist_ok=True)


### set a directory for the master boundary data
master_bound_final_dir = os.path.join(bound_final_dir, 'master_boundary')
os.makedirs(master_bound_final_dir, exist_ok=True)


### set a directory for the federal boundary data
fed_bound_final_dir = os.path.join(bound_final_dir, 'federal_boundary')
os.makedirs(fed_bound_final_dir, exist_ok=True)


### set a directory for the state boundary data
state_bound_final_dir = os.path.join(bound_final_dir, 'state_boundary')
os.makedirs(state_bound_final_dir, exist_ok=True)

### Save Pawnee West processed data in these folders

In [23]:
### save master boundary
pawnee_west_path = os.path.join(master_bound_final_dir, "pawnee_master_west.shp")
pawnee_west.to_file(pawnee_west_path)

### save fed boundary
pawnee_fed_west_path = os.path.join(fed_bound_final_dir, "pawnee_fed_west.shp")
pawnee_fed_west.to_file(pawnee_fed_west_path)

### save state boundary
pawnee_state_west_path = os.path.join(state_bound_final_dir, "pawnee_state_west.shp")
pawnee_state_west.to_file(pawnee_state_west_path)

INFO:Created 1 records
INFO:Created 1 records


c:\Users\warno\miniconda3\envs\earth-analytics-python\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field SALEDT created as String field, though DateTime requested.
  ogr_write(
INFO:Created 53 records


### Universal Calls for Pawnee West Boundary Data

In [ ]:
### root dir
data_dir = os.path.join(repo_root, 'data')

### main boundary dir
boundary_dir = os.path.join(data_dir, 'boundaries')

### boundary dir for processed data for Western Pawnee
boundary_dir_west = os.path.join(boundary_dir, 'boundary-data-final-west')


### MASTER
### master boundary dir
master_bound_west = os.path.join(boundary_dir_west, 'master_boundary')

### master boundary shapefile
master_bound_west_path = os.path.join(master_bound_west, "pawnee_master_west.shp")

### master boundary convert to gdf
master_bound_west_gdf = gpd.read_file(master_bound_west_path)


### FEDERAL
### federal boundary dir
federal_bound_west = os.path.join(boundary_dir_west, 'federal_boundary')

### federal boundary shapefile 
federal_bound_west_path = os.path.join(federal_bound_west, 'pawnee_fed_west.shp')

### federal boundary convert to gdf
federal_bound_west_gdf = gpd.read_file(federal_bound_west_path)


### STATE
### state boundary dir
state_bound_west = os.path.join(boundary_dir_west, 'state_boundary')

### state boundary shapefile 
state_bound_west_path = os.path.join(state_bound_west, 'pawnee_state_west.shp')

### state boundary convert to gdf
state_bound_west_gdf = gpd.read_file(state_bound_west_path)